# Lecture 09 — LLM-driven extraction

> *"Selectors are a contract with the page. LLMs are a negotiation."*

Up until lecture 08 every extraction step in this course has been deterministic: a CSS selector or an XPath says "the price lives at `span.price`", you trust the page, and you parse. That works *beautifully* on structured product pages, listings, and APIs. It falls apart the moment the data is **free-form prose**, **inconsistent across pages**, or **silently re-skinned** every quarter.

This lecture is about when to swap your selectors for a language model — and, just as importantly, when not to.

## 1. Where selectors break

A few real-world shapes that make a CSS-selector approach miserable:

- **Heterogeneous templates.** A jobs board where every employer customises the listing layout. Same site, fifteen DOM shapes.
- **Prose with embedded facts.** Press releases, court opinions, regulatory filings. The data you want ("the company raised $40M") is buried in a sentence, not a `<dd>`.
- **Schema drift.** A site you scrape weekly subtly renames `data-test="price-now"` to `data-test="price"`. Your selector silently returns nothing.
- **PDF-to-HTML conversions.** Layouts come out as a soup of absolutely-positioned `<div>`s with no semantic structure.

In each case the pattern "locate node, read text" doesn't quite fit. You instead want: "read this whole blob and tell me the structured facts."

## 2. What an LLM is good at here

Given a page (or a chunk of one) and a target schema, a modern LLM is *very* good at:

- Pulling typed facts out of free text ("valuation", "close date", "stage").
- Normalising messy values ("$40M", "40 million USD", "forty million dollars" → `40_000_000`).
- Tolerating layout drift — it doesn't care whether your value is in a `<td>` or a `<p>`.
- Saying "not present" when the field genuinely isn't there.

It is *bad* at:

- Acting as a faster CSS selector. If you already know the data lives at `span.price`, calling an LLM is the wrong tool.
- Being deterministic. The same page can yield slightly different outputs across calls.
- Being cheap. Even a small page is hundreds to thousands of tokens. At scale that bill is real.

## 3. The pattern

The pipeline that works in production looks like this:

```
fetch  →  pre-clean  →  LLM extract  →  validate  →  store
```

Let's unpack each step:

1. **Fetch** — still a normal `httpx` GET (lecture 02), or a Playwright render if the page is JS-heavy (lecture 05).
2. **Pre-clean** — strip nav, footers, ads, scripts. Send the model the *meaningful* content only. Tokens cost money; nav doesn't.
3. **LLM extract** — one call per page (or per chunk), with a *strict schema* attached.
4. **Validate** — parse the model's output through your schema. Reject anything that doesn't fit. Never trust the wire.
5. **Store** — plain SQLite is plenty for almost everything (lecture 10).

The critical move is step 4. Treat the LLM as an untrusted parser. Validate, don't believe.

## 4. Pre-cleaning: the cheap win

The single biggest cost lever in LLM extraction is **how many tokens you send**. A typical product page is ~30KB of HTML, of which maybe ~3KB is the actual content you care about. The other 27KB is nav, footer, tracking script, and recommendation widgets.

A reasonable pre-clean:

In [ ]:
from bs4 import BeautifulSoup

DROP_TAGS = ("script", "style", "nav", "footer", "header", "aside", "noscript")

def cleaned_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(DROP_TAGS):
        tag.decompose()
    # If you know a content selector, prefer it.
    main = soup.select_one("main, article, [role=main]") or soup.body or soup
    text = main.get_text(" ", strip=True)
    # Collapse whitespace.
    return " ".join(text.split())


Send `cleaned_text(html)` to the model rather than raw HTML. You'll typically cut input tokens by 5–10x and *improve* extraction quality at the same time — there's less for the model to be distracted by.

## 5. A worked extraction (Claude)

We'll use Anthropic's SDK with a strict JSON schema. Pydantic is the most ergonomic way to express the schema and validate the response in one step.

In [ ]:
# pip install anthropic pydantic
from pydantic import BaseModel, Field, ValidationError
from anthropic import Anthropic
import json

client = Anthropic()  # picks up ANTHROPIC_API_KEY

class FundingRound(BaseModel):
    company: str
    amount_usd: int | None = Field(description="Total raise in USD, integer. None if unstated.")
    stage: str | None = Field(description="e.g. 'Seed', 'Series A'. None if unstated.")
    announced_on: str | None = Field(description="ISO date YYYY-MM-DD. None if unstated.")
    lead_investor: str | None = None

SYSTEM = (
    "You extract structured facts from press releases. "
    "Return JSON matching the requested schema. "
    "Use null for any field that is not clearly stated. Do not invent values."
)

def extract_round(article_text: str) -> FundingRound | None:
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=SYSTEM,
        messages=[{
            "role": "user",
            "content": (
                f"Extract a funding round from this article. "
                f"Schema: {FundingRound.model_json_schema()}\n\n"
                f"Article:\n{article_text}"
            ),
        }],
    )
    raw = msg.content[0].text.strip()
    # Strip ```json fences if the model added them.
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[1].rsplit("\n", 1)[0]
    try:
        return FundingRound.model_validate_json(raw)
    except ValidationError as e:
        print("Validation failed:", e)
        return None


Notice the discipline:

- The schema is the source of truth, not the prompt.
- The model is told explicitly: use `null`, don't invent.
- The response is validated before it's allowed anywhere near your storage.
- We pick **Haiku 4.5**, not Opus. Extraction is a small, well-scoped task; the cheapest capable model is almost always the right call.

## 6. Costs, latency, and prompt caching

A realistic order of magnitude for the call above (clean article ~2K tokens in, ~150 tokens out, Haiku 4.5):

- **Latency:** ~1–3 seconds per page.
- **Cost:** sub-cent per page.
- **Throughput:** trivially parallelisable since each call is independent (lecture 06's politeness still applies to the *source site*, not the model API).

If you're processing many pages that share a long, stable system prompt or schema, turn on **prompt caching** — Anthropic's SDK supports `cache_control` markers that let you reuse the same prefix across calls at a heavy discount. For a high-volume scraper extracting against the same schema, this is often a 5–10x effective cost reduction.

The rule of thumb: if you'd send the same chunk of prompt more than ~3 times in a 5-minute window, cache it.

## 7. When *not* to use an LLM

- **Structured pages with stable selectors.** A `span.price` on a product page is faster, cheaper, and 100% deterministic. Don't reach for the model.
- **High-volume, low-margin scrapes.** A million pages a day at 0.5¢ each is $5,000/day. A selector-based scrape is ~free. Do the maths first.
- **Adversarial extraction.** If the site is actively changing layouts to thwart you, you have a different problem (lecture 08).
- **Anything where wrongness is dangerous.** Medical, legal, financial extraction without a human-in-the-loop is asking for trouble. The model will sometimes be confidently wrong.

A useful heuristic: *if a competent intern with the page open could write down the answer in five seconds, an LLM probably can too. If the intern would need to make judgement calls, so does the model — and "judgement call" means "sometimes wrong".*

## 8. Keeping the pipeline honest

A few small habits that pay off enormously over time:

- **Log every extraction.** Store `(url, html_hash, schema_version, raw_response, parsed_value)`. When something goes wrong six months later, you'll bless yourself.
- **Sample-audit weekly.** Pull 20 random extractions, eyeball them. You will find drift.
- **Version your schema.** When you add a field, bump the schema version. Don't silently mix shapes.
- **Have a `confidence` field.** Ask the model to flag when it's guessing. It will, often correctly, and that gives you a queue for human review.

## Recap

- LLM extraction shines on heterogeneous, prose-y, drift-prone pages — not on neat product grids.
- The pipeline is: fetch → pre-clean → model call with a strict schema → validate → store.
- Pre-cleaning the HTML is the single biggest cost lever. Don't send nav.
- Pick the smallest capable model (Haiku 4.5 over Opus for plain extraction). Cache prompts when you're sending the same prefix repeatedly.
- Treat the model as an untrusted parser: validate every response.
- Don't reach for an LLM when a selector will do, when volume is huge and margins thin, or when wrongness is dangerous.

## Exercises

1. Take any 3 funding-announcement pages from TechCrunch or a press-release wire. Run the `extract_round` function above against each. Where it fails, ask: was it the model, the prompt, or your schema being too strict?
2. Add a `confidence: float` field (0.0–1.0) to `FundingRound` and instruct the model to populate it. Run on 10 pages. Sort by confidence ascending; eyeball the bottom 3. Were they actually the hardest cases?
3. Time how long the *fetch* takes vs. the *LLM call* for one page. Which dominates? What does that tell you about where to spend optimisation effort?
4. Take a page where selectors *would* work cleanly (any product page on a major retailer). Build both a CSS-selector extractor and an LLM extractor. Compare on cost, latency, and code length. Which would you ship?

## Up next

**Lecture 10** — the capstone. Taking everything from lectures 00–09 and assembling it into a single, dependable, *boring* pipeline you'd actually be willing to leave running unattended.